# End-to-End Pipeline Test

Exercises every service built so far in a single linear pipeline:

1. **Arxiv client** → fetch paper metadata
2. **PDF parser (Docling)** → download + parse PDF
3. **Repository (Postgres)** → persist metadata
4. **TextChunker** → split parsed text into chunks
5. **OpenAI embeddings** → vectorize each chunk
6. **OpenSearch client** → create index, bulk-write chunks
7. **Search** → BM25, vector, and hybrid queries

Run cells top to bottom. Each cell prints what it produced so you can verify before continuing.

**Prerequisites:**
- Postgres + Airflow containers running (`docker compose up -d postgres airflow`).
- OpenSearch container running (`docker compose up -d opensearch`).
- `OPENAI_API_KEY` in `.env`.

## 0. Setup

Locate the project root, add it to `sys.path`, configure logging so we can see what services are doing.

In [1]:
import logging
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
print(f"Project root: {project_root}")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
    force=True,
)

Project root: /Users/kumarrohit/Arxiv_Paper_Curator


## 1. Build all services

Every service is constructed via its factory. Each factory is `@lru_cache`d, so calling these is idempotent and cheap.

Important: import `Paper` **before** `make_database()` so the ORM table registers against `Base` before `create_all` runs.

In [3]:
from src.models.paper import Paper  # MUST be imported before make_database()
from src.db.factory import make_database
from src.services.arxiv.factory import make_arxiv_client
from src.services.pdf_parser.factory import make_pdf_parser_service
from src.services.opensearch.factory import make_opensearch_client
from src.services.embeddings.factory import make_openai_embeddings_client
from src.services.indexing.text_chunker import TextChunker
from src.services.agents.factory import make_agentic_rag
from src.repositories.paper import PaperRepository

db        = make_database()
arxiv     = make_arxiv_client()
parser    = make_pdf_parser_service()       # first call downloads Docling models (~1GB, slow!)
opensearch = make_opensearch_client()
embedder  = make_openai_embeddings_client()
agentic_rag = make_agentic_rag()
chunker   = TextChunker()                    # no factory yet — built directly

print("All services built.")

2026-07-12 13:21:06,354 | INFO    | src.db.interfaces.postgresql | Attempting to connect to PostgreSQL at: localhost:5432/rag_db
2026-07-12 13:21:06,521 | INFO    | src.db.interfaces.postgresql | Database connection test successful
2026-07-12 13:21:06,576 | INFO    | src.db.interfaces.postgresql | All tables already exist - no new tables created
2026-07-12 13:21:06,576 | INFO    | src.db.interfaces.postgresql | PostgreSQL database initialized successfully
2026-07-12 13:21:06,576 | INFO    | src.db.interfaces.postgresql | Database: rag_db
2026-07-12 13:21:06,576 | INFO    | src.db.interfaces.postgresql | Total tables: papers
2026-07-12 13:21:06,576 | INFO    | src.db.interfaces.postgresql | Database connection established
2026-07-12 13:21:06,647 | INFO    | src.services.opensearch.client | OpenSearch client initialized with host: http://localhost:9200
2026-07-12 13:21:06,741 | INFO    | src.services.embeddings.openai_client | OpenAI embeddings client initialized: model=text-embedding-3-

All services built.


## 2. Health checks

Confirm each external system is reachable before doing real work. If any of these fail, fix that before continuing.

In [5]:
from sqlalchemy import text

# Postgres
with db.get_session() as session:
    pg_ok = session.execute(text("SELECT 1")).scalar() == 1
print(f"Postgres reachable: {pg_ok}")

# OpenSearch
print(f"OpenSearch healthy: {opensearch.health_check()}")

# OpenAI
vec = await embedder.embed_text("healthcheck")
print(f"OpenAI reachable: {len(vec) == embedder.settings.dimensions} ({len(vec)} dims)")

2026-07-12 13:24:33,202 | INFO    | opensearch | GET http://localhost:9200/_cluster/health [status:200 request:0.062s]


Postgres reachable: True
OpenSearch healthy: True


2026-07-12 13:24:33,968 | INFO    | httpx | HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-07-12 13:24:33,979 | INFO    | src.services.embeddings.openai_client | Embedded 1 texts in 1 batch(es)


OpenAI reachable: True (1024 dims)


## 3. Setup OpenSearch index + RRF pipeline

Creates the `arxiv-papers-chunks` index (with the hybrid mapping) and registers the RRF search pipeline. Idempotent — safe to re-run.

In [7]:
setup_results = opensearch.setup_indices(force=True)
print("Setup results:", setup_results)
print("Index stats:  ", opensearch.get_index_stats())

2026-07-12 13:24:59,899 | INFO    | opensearch | HEAD http://localhost:9200/arxiv-papers-chunks [status:200 request:0.043s]
2026-07-12 13:25:00,038 | INFO    | opensearch | DELETE http://localhost:9200/arxiv-papers-chunks [status:200 request:0.137s]
2026-07-12 13:25:00,038 | INFO    | src.services.opensearch.client | Deleted existing hybrid index: arxiv-papers-chunks
2026-07-12 13:25:00,149 | INFO    | opensearch | PUT http://localhost:9200/arxiv-papers-chunks [status:200 request:0.103s]
2026-07-12 13:25:00,149 | INFO    | src.services.opensearch.client | Created hybrid index: arxiv-papers-chunks
2026-07-12 13:25:00,152 | WARNING | opensearch | GET http://localhost:9200/_ingest/pipeline/hybrid-rrf-pipeline [status:404 request:0.003s]
2026-07-12 13:25:00,154 | WARNING | opensearch | GET http://localhost:9200/_ingest/pipeline/hybrid-rrf-pipeline [status:404 request:0.001s]
2026-07-12 13:25:00,168 | INFO    | opensearch | PUT http://localhost:9200/_search/pipeline/hybrid-rrf-pipeline [sta

Setup results: {'hybrid_index': True, 'rrf_pipeline': True}
Index stats:   {'index_name': 'arxiv-papers-chunks', 'exists': True, 'document_count': 0, 'deleted_count': 0, 'size_in_bytes': 208}


## 4. Fetch one paper from arxiv

Calls the arxiv API and gets recent cs.AI papers. Limited to 1 so we don't burn through rate limits.

In [8]:

papers = await arxiv.fetch_papers(max_results=10)
if not papers:
    raise RuntimeError("No papers returned — arxiv may be rate-limiting. Wait a few minutes.")

paper = papers[0]
print(f"arxiv_id : {paper.arxiv_id}")
print(f"title    : {paper.title}")
print(f"authors  : {paper.authors[:3]} ...")
print(f"published: {paper.published_date}")
print(f"pdf_url  : {paper.pdf_url}")

2026-07-12 13:25:23,218 | INFO    | src.services.arxiv.client | Fetching 10 cs.AI papers from arxiv
2026-07-12 13:25:23,567 | INFO    | httpx | HTTP Request: GET https://export.arxiv.org/api/query?search_query=cat:cs.AI&start=0&max_results=10&sortBy=submittedDate&sortOrder=descending "HTTP/1.1 200 OK"
2026-07-12 13:25:23,593 | INFO    | src.services.arxiv.client | Fetched 10 papers.


arxiv_id : 2607.08763v1
title    : OpenCoF: Learning to Reason Through Video Generation
authors  : ['Xinyan Chen', 'Ziyu Guo', 'Renrui Zhang'] ...
published: 2026-07-09T17:58:29Z
pdf_url  : https://arxiv.org/pdf/2607.08763v1


## 5. Download + parse the PDF

Downloads the PDF (cached locally on second run), then Docling extracts structured content. **First run is slow** — Docling downloads model weights.

In [9]:
pdf_path = await arxiv.download_pdf(paper)
print(f"PDF saved to: {pdf_path}")

parsed = await parser.parse_pdf(pdf_path)
print(f"Raw text length: {len(parsed.raw_text)} chars")
print(f"Sections found : {len(parsed.sections)}")
for s in parsed.sections[:5]:
    print(f"  - {s.title!r:50}  ({len(s.content.split())} words)")
print(f"  ... and {max(0, len(parsed.sections) - 5)} more")

2026-07-12 13:25:42,245 | INFO    | src.services.arxiv.client | Downloading PDF from https://arxiv.org/pdf/2607.08763v1
2026-07-12 13:25:45,323 | INFO    | httpx | HTTP Request: GET https://arxiv.org/pdf/2607.08763v1 "HTTP/1.1 200 OK"
2026-07-12 13:25:45,479 | INFO    | src.services.arxiv.client | Successfully downloaded to 2607.08763v1.pdf
2026-07-12 13:25:45,599 | INFO    | docling.datamodel.document | detected formats: [<InputFormat.PDF: 'pdf'>]


PDF saved to: data/arxiv_pdfs/2607.08763v1.pdf


2026-07-12 13:25:45,712 | INFO    | docling.document_converter | Going to convert document batch...
2026-07-12 13:25:45,725 | INFO    | docling.document_converter | Initializing pipeline for StandardPdfPipeline with options hash af605ac4b878857c2150ff1e95ea3efd
2026-07-12 13:25:45,879 | INFO    | docling.models.factories.base_factory | Loading plugin 'docling_defaults'
2026-07-12 13:25:45,909 | INFO    | docling.models.factories | Registered picture descriptions: ['picture_description_vlm_engine', 'vlm', 'api']
2026-07-12 13:25:45,933 | INFO    | docling.models.factories.base_factory | Loading plugin 'docling_defaults'
2026-07-12 13:25:46,056 | INFO    | docling.models.factories | Registered ocr engines: ['auto', 'easyocr', 'kserve_v2_ocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2026-07-12 13:25:46,065 | INFO    | docling.models.factories.base_factory | Loading plugin 'docling_defaults'
2026-07-12 13:25:46,076 | INFO    | docling.models.factories | Registered layout engines: [

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

2026-07-12 13:25:49,297 | INFO    | docling.models.factories.base_factory | Loading plugin 'docling_defaults'
2026-07-12 13:25:49,307 | INFO    | docling.models.factories | Registered table structure engines: ['docling_tableformer', 'docling_tableformer_v2', 'granite_vision_table']
2026-07-12 13:25:49,541 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/models/docling-project/docling-models/revision/v2.3.0 "HTTP/1.1 200 OK"
2026-07-12 13:25:50,555 | INFO    | docling.utils.accelerator_utils | Accelerator device: 'mps'
2026-07-12 13:25:51,279 | INFO    | docling.pipeline.base_pipeline | Processing document 2607.08763v1.pdf
2026-07-12 13:26:23,523 | INFO    | docling.document_converter | Finished converting document 2607.08763v1.pdf in 37.98 sec.
2026-07-12 13:26:23,615 | INFO    | src.services.pdf_parser.parser | Parsed 2607.08763v1.pdf


Raw text length: 61339 chars
Sections found : 50
  - 'Content'                                           (5 words)
  - 'OpenCoF: Learning to Reason Through Video Generation'  (47 words)
  - 'Abstract'                                          (220 words)
  - '1 Introduction'                                    (880 words)
  - '2 OpenCoF-17K'                                     (55 words)
  ... and 45 more


## 6. Store in Postgres

Use the repository's `upsert` — idempotent, so re-running this cell with the same paper updates instead of duplicating.

In [10]:
from datetime import datetime
from src.schemas.arxiv.paper import PaperCreate

paper_create = PaperCreate(
    arxiv_id=paper.arxiv_id,
    title=paper.title,
    authors=paper.authors,
    abstract=paper.abstract,
    categories=paper.categories,
    published_date=datetime.fromisoformat(paper.published_date.replace("Z", "+00:00")),
    pdf_url=paper.pdf_url,
    raw_text=parsed.raw_text,
    sections=[s.model_dump() for s in parsed.sections],
    parser_used="docling",
    pdf_processed=True,
    pdf_processing_date=datetime.utcnow(),
)

with db.get_session() as session:
    repo = PaperRepository(session)
    stored = repo.upsert(paper_create)
    stored_id = stored.id  # capture before session closes
    total = repo.get_count()

print(f"Stored paper id: {stored_id}")
print(f"Total papers in DB: {total}")

Stored paper id: 54e77898-9114-4885-a42b-eb981a030955
Total papers in DB: 4


/var/folders/bn/q6q84kl961g01gkjcnxj1jk80000gn/T/ipykernel_84656/2421299209.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  pdf_processing_date=datetime.utcnow(),


## 7. Chunk the paper

Use the section-aware chunker. With sections present it uses the hybrid strategy; without sections it falls back to word-based chunking.

In [11]:
sections_dict = {s.title: s.content for s in parsed.sections}

chunks = chunker.chunk_paper(
    title=paper.title,
    abstract=paper.abstract,
    full_text=parsed.raw_text,
    arxiv_id=paper.arxiv_id,
    paper_id=str(stored_id),
    sections=sections_dict,
)

print(f"Produced {len(chunks)} chunks")
for i, c in enumerate(chunks[:5]):
    print(f"  chunk {i}: {c.metadata.word_count:>4} words  |  section: {c.metadata.section_title!r}")
print(f"  ... and {max(0, len(chunks) - 5)} more")

2026-07-12 13:27:05,882 | INFO    | src.services.indexing.text_chunker | Chunked paper 2607.08763v1: 883 words -> 2 chunks
2026-07-12 13:27:05,887 | INFO    | src.services.indexing.text_chunker | Chunked paper 2607.08763v1: 1716 words -> 4 chunks
2026-07-12 13:27:05,888 | INFO    | src.services.indexing.text_chunker | Created 24 section-based chunks for 2607.08763v1


Produced 24 chunks
  chunk 0:  268 words  |  section: 'Abstract + Combined'
  chunk 1:  607 words  |  section: '1 Introduction (Part 1)'
  chunk 2:  448 words  |  section: '1 Introduction (Part 2) + Combined'
  chunk 3:  345 words  |  section: '2.1 Dataset Statistics + Combined'
  chunk 4:  144 words  |  section: '2.2.1 Instance-based Rendering'
  ... and 19 more


## 8. Embed the chunks

Single `embed_batch` call — auto-batches into groups of `settings.batch_size` (default 100). For ~20 chunks that's 1 API call.

Costs roughly **$0.0001** for a typical paper.

In [12]:
chunk_texts = [c.text for c in chunks]
vectors = await embedder.embed_batch(chunk_texts)

print(f"Embedded {len(vectors)} chunks")
print(f"Each vector has {len(vectors[0])} dimensions")
print(f"First vector starts: {vectors[0][:4]}")
assert len(vectors) == len(chunks), "Embed count must match chunk count!"

2026-07-12 13:27:13,069 | INFO    | httpx | HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-07-12 13:27:13,322 | INFO    | src.services.embeddings.openai_client | Embedded 24 texts in 1 batch(es)


Embedded 24 chunks
Each vector has 1024 dimensions
First vector starts: [0.039459228515625, 0.019561767578125, 0.006374359130859375, 0.0078277587890625]


## 9. Index the chunks in OpenSearch

Convert each (`TextChunk`, `embedding`) into the dict shape `bulk_index_chunks` expects. We fill in the metadata fields the index mapping requires.

In [13]:
from datetime import datetime

chunks_for_indexing = []
now_iso = datetime.utcnow().isoformat()

for chunk, vector in zip(chunks, vectors):
    chunk_data = {
        "chunk_id":         f"{paper.arxiv_id}_{chunk.metadata.chunk_index}",
        "arxiv_id":         paper.arxiv_id,
        "paper_id":         str(stored_id),
        "chunk_index":      chunk.metadata.chunk_index,
        "chunk_text":       chunk.text,
        "chunk_word_count": chunk.metadata.word_count,
        "start_char":       chunk.metadata.start_char,
        "end_char":         chunk.metadata.end_char,
        "title":            paper.title,
        "authors":          ", ".join(paper.authors),
        "abstract":         paper.abstract,
        "categories":       paper.categories,
        "published_date":   paper.published_date,
        "section_title":    chunk.metadata.section_title or "unknown",
        "embedding_model":  embedder.settings.model,
        "created_at":       now_iso,
        "updated_at":       now_iso,
    }
    chunks_for_indexing.append({"chunk_data": chunk_data, "embedding": vector})

result = opensearch.bulk_index_chunks(chunks_for_indexing)
print("Bulk index result:", result)
print("Index stats now:  ", opensearch.get_index_stats())

/var/folders/bn/q6q84kl961g01gkjcnxj1jk80000gn/T/ipykernel_84656/4032541809.py:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now_iso = datetime.utcnow().isoformat()
2026-07-12 13:27:18,692 | INFO    | opensearch | POST http://localhost:9200/_bulk?refresh=true [status:200 request:1.227s]
2026-07-12 13:27:18,694 | INFO    | src.services.opensearch.client | Bulk indexed 24 chunks, 0 failed
2026-07-12 13:27:18,707 | INFO    | opensearch | HEAD http://localhost:9200/arxiv-papers-chunks [status:200 request:0.011s]
2026-07-12 13:27:18,717 | INFO    | opensearch | GET http://localhost:9200/arxiv-papers-chunks/_stats [status:200 request:0.010s]


Bulk index result: {'success': 24, 'failed': 0}
Index stats now:   {'index_name': 'arxiv-papers-chunks', 'exists': True, 'document_count': 24, 'deleted_count': 0, 'size_in_bytes': 546167}


## 10. Search the index — BM25 (keyword)

Pure keyword search. Pick a word that should appear in the paper's content.

In [11]:
# Use a word from the paper's title so we know it should match.
keyword = paper.title.split()[0]
print(f"Searching for keyword: {keyword!r}\n")

results = opensearch.search_papers(query=keyword, size=3)
print(f"Total hits: {results['total']}")
for h in results["hits"]:
    print(f"  score={h['score']:.3f}  section={h.get('section_title')}")
    print(f"  text: {h['chunk_text'][:120]}...")
    print()

Searching for keyword: 'How'



2026-06-08 22:21:15,656 | INFO    | opensearch | POST http://localhost:9200/arxiv-papers-chunks/_search [status:200 request:1.322s]
2026-06-08 22:21:15,658 | INFO    | src.services.opensearch.client | BM25 search for 'How...' returned 45 results


Total hits: 45
  score=0.055  section=HOW RELIABLE ARE LLMS WHEN IT COMES TO PLAYING DICE?
  text: How reliable are LLMs when it comes to playing dice?

Abstract: We investigate the probabilistic reasoning capabilities ...

  score=0.055  section=HOW RELIABLE ARE LLMS WHEN IT COMES TO PLAYING DICE?
  text: How reliable are LLMs when it comes to playing dice?

Abstract: We investigate the probabilistic reasoning capabilities ...

  score=0.055  section=HOW RELIABLE ARE LLMS WHEN IT COMES TO PLAYING DICE?
  text: How reliable are LLMs when it comes to playing dice?

Abstract: We investigate the probabilistic reasoning capabilities ...



## 11. Search the index — pure vector (semantic)

Embed a natural-language query, then ask OpenSearch for nearest-neighbor chunks by vector similarity.

In [13]:
query = "main contribution of this paper"
query_vec = await embedder.embed_text(query)

results = opensearch.search_chunks_vector(query_embedding=query_vec, size=3)
print(f"Query: {query!r}")
print(f"Total hits: {results['total']}\n")
for h in results["hits"]:
    print(f"  score={h['score']:.3f}  section={h.get('section_title')}")
    print(f"  text: {h['chunk_text'][:120]}...")
    print()

2026-06-08 22:21:58,757 | INFO    | httpx | HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-08 22:21:58,768 | INFO    | src.services.embeddings.openai_client | Embedded 1 texts in 1 batch(es)
2026-06-08 22:21:59,335 | INFO    | opensearch | POST http://localhost:9200/arxiv-papers-chunks/_search [status:200 request:0.563s]


Query: 'main contribution of this paper'
Total hits: 3

  score=0.676  section=4 Discussion
  text: How reliable are LLMs when it comes to playing dice?

Abstract: We investigate the probabilistic reasoning capabilities ...

  score=0.676  section=4 Discussion
  text: How reliable are LLMs when it comes to playing dice?

Abstract: We investigate the probabilistic reasoning capabilities ...

  score=0.676  section=4 Discussion
  text: How reliable are LLMs when it comes to playing dice?

Abstract: We investigate the probabilistic reasoning capabilities ...



## 12. Search the index — hybrid (BM25 + vector + RRF)

The headline feature. Same query, but OpenSearch runs BOTH retrievers and fuses the rankings via the RRF pipeline.

In [14]:
query = "main contribution of this paper"
query_vec = await embedder.embed_text(query)

results = opensearch.search_chunks_hybrid(
    query=query,
    query_embedding=query_vec,
    size=3,
)
print(f"Query: {query!r}")
print(f"Total hits: {results['total']}\n")
for h in results["hits"]:
    print(f"  score={h['score']:.4f}  section={h.get('section_title')}")
    print(f"  text: {h['chunk_text'][:120]}...")
    print()

2026-06-08 22:22:13,336 | INFO    | httpx | HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-08 22:22:13,338 | INFO    | src.services.embeddings.openai_client | Embedded 1 texts in 1 batch(es)
2026-06-08 22:22:14,088 | INFO    | opensearch | POST http://localhost:9200/arxiv-papers-chunks/_search?search_pipeline=hybrid-rrf-pipeline [status:200 request:0.739s]
2026-06-08 22:22:14,090 | INFO    | src.services.opensearch.client | Native hybrid search for 'main contribution of this paper...' returned 3 results


Query: 'main contribution of this paper'
Total hits: 3

  score=0.0164  section=1 Introduction
  text: How reliable are LLMs when it comes to playing dice?

Abstract: We investigate the probabilistic reasoning capabilities ...

  score=0.0164  section=4 Discussion
  text: How reliable are LLMs when it comes to playing dice?

Abstract: We investigate the probabilistic reasoning capabilities ...

  score=0.0161  section=1 Introduction
  text: How reliable are LLMs when it comes to playing dice?

Abstract: We investigate the probabilistic reasoning capabilities ...



## 13. (Optional) Inspect everything you wrote

Lists Postgres rows and OpenSearch chunks for the paper you just processed. Useful for sanity-checking that all stages stored what they should have.

In [15]:
print("=== Postgres ===")
with db.get_session() as session:
    repo = PaperRepository(session)
    row = repo.get_by_arxiv_id(paper.arxiv_id)
    print(f"  arxiv_id={row.arxiv_id}  pdf_processed={row.pdf_processed}  sections={len(row.sections or [])}")

print("\n=== OpenSearch ===")
os_chunks = opensearch.get_chunks_by_paper(paper.arxiv_id)
print(f"  Found {len(os_chunks)} chunks for {paper.arxiv_id}")
for c in os_chunks[:3]:
    print(f"    chunk_index={c['chunk_index']}  section={c.get('section_title')}  words={c.get('chunk_word_count')}")

=== Postgres ===
  arxiv_id=2606.07515v1  pdf_processed=True  sections=17

=== OpenSearch ===


2026-06-08 22:22:23,725 | INFO    | opensearch | POST http://localhost:9200/arxiv-papers-chunks/_search [status:200 request:0.217s]


  Found 45 chunks for 2606.07515v1
    chunk_index=0  section=HOW RELIABLE ARE LLMS WHEN IT COMES TO PLAYING DICE?  words=167
    chunk_index=0  section=HOW RELIABLE ARE LLMS WHEN IT COMES TO PLAYING DICE?  words=167
    chunk_index=0  section=HOW RELIABLE ARE LLMS WHEN IT COMES TO PLAYING DICE?  words=167


## 14. (Optional) Cleanup

Removes the chunks you just indexed from OpenSearch (Postgres row is left alone — you can re-upsert anytime). Useful when you're iterating and don't want stale data.

In [16]:
deleted = opensearch.delete_paper_chunks(paper.arxiv_id)
print(f"Deleted chunks for {paper.arxiv_id}: {deleted}")
print("Stats now:", opensearch.get_index_stats())

2026-06-08 22:22:42,003 | INFO    | opensearch | POST http://localhost:9200/arxiv-papers-chunks/_delete_by_query?refresh=true [status:200 request:0.473s]
2026-06-08 22:22:42,012 | INFO    | src.services.opensearch.client | Deleted 45 chunks for paper 2606.07515v1
2026-06-08 22:22:42,022 | INFO    | opensearch | HEAD http://localhost:9200/arxiv-papers-chunks [status:200 request:0.008s]
2026-06-08 22:22:42,143 | INFO    | opensearch | GET http://localhost:9200/arxiv-papers-chunks/_stats [status:200 request:0.120s]


Deleted chunks for 2606.07515v1: True
Stats now: {'index_name': 'arxiv-papers-chunks', 'exists': True, 'document_count': 0, 'deleted_count': 75, 'size_in_bytes': 1056688}


## What this notebook proved

If every cell ran without errors and the search cells returned the paper's chunks, then the full pipeline works:

```
arxiv API  →  ArxivClient.fetch_papers()        ✓
           →  ArxivClient.download_pdf()         ✓
           →  PDFParserService.parse_pdf()       ✓  (Docling)
           →  PaperRepository.upsert()           ✓  (Postgres)
           →  TextChunker.chunk_paper()          ✓
           →  OpenAIEmbeddingsClient.embed_batch() ✓
           →  OpenSearchClient.bulk_index_chunks() ✓
           →  OpenSearchClient.search_*()        ✓  (BM25, vector, hybrid)
```

**What's NOT in this notebook yet:**
- The Airflow DAG task that wraps all this (the empty `indexing.py`).
- A `HybridIndexingService` that composes the chunker + embedder + opensearch into one `index_paper(paper)` method.

Both are mechanical wrap-ups of what this notebook already does — no new concepts. Once they're built, the DAG can run this whole flow per paper automatically.